# DISCOVERING AI AGENT — Notebook thực hành
**Mã bài:** agent01_CT_04_hainn · **Ngày:** 23/10/2026

Triển khai tự viết dựa trên mục **Goal-based agents**, **Reasoning with available tools**
và **Activity logs** trong [IBM, What are AI agents?](https://www.ibm.com/think/topics/ai-agents).
Đây là agent ký hiệu dùng BFS, **không gọi LLM, không huấn luyện RL, không phải bản tái lập ReAct**.
Đồ thị, biến cố và ca kiểm tra là dữ liệu mô phỏng phục vụ học tập.
Chạy các cell từ trên xuống; chỉ dùng thư viện chuẩn Python.

In [1]:
from collections import deque
import platform

GRAPH = {
    "A": ["B", "C"], "B": ["A", "D", "C"],
    "C": ["A", "B", "E"], "D": ["B", "F"],
    "E": ["C", "F"], "F": ["D", "E"],
}

def edge(a, b):
    return frozenset((a, b))

print("Python:", platform.python_version())
print("Nodes:", len(GRAPH))
print("Goal: A -> F; all edges have unit cost.")

Python: 3.9.6
Nodes: 6
Goal: A -> F; all edges have unit cost.


## 1. Bộ lập kế hoạch
BFS trả về đường ngắn nhất trên **mô hình hiện biết**. `blocked` là bộ nhớ về các cạnh
không đi được. Nếu không có đường, trả về `None`; tập `seen` ngăn duyệt lặp.

In [2]:
def plan_route(graph, start, goal, blocked):
    queue = deque([[start]])
    seen = {start}
    while queue:
        path = queue.popleft()
        node = path[-1]
        if node == goal:
            return path
        for nxt in graph.get(node, []):
            if nxt not in seen and edge(node, nxt) not in blocked:
                seen.add(nxt)
                queue.append(path + [nxt])
    return None

print("Initial plan:", plan_route(GRAPH, "A", "F", set()))
print("Known closure:", plan_route(GRAPH, "B", "F", {edge("B", "D")}))

Initial plan: ['A', 'B', 'D', 'F']
Known closure: ['B', 'C', 'E', 'F']


## 2. Công cụ tương tác môi trường
Sau lần di chuyển đầu tiên, môi trường đóng cạnh B–D. Agent chỉ phát hiện khi thử
đi qua cạnh đó; kế hoạch ban đầu không được biết trước biến cố. `move` là công cụ
duy nhất làm thay đổi vị trí và trả về quan sát có cấu trúc.

In [3]:
class World:
    def __init__(self, start="A", dynamic=False, closed=()):
        self.pos = start
        self.dynamic = dynamic
        self.closed = set(closed)
        self.moves = 0

    def move(self, nxt):
        if self.dynamic and self.moves >= 1:
            self.closed.add(edge("B", "D"))
        if nxt not in GRAPH.get(self.pos, []):
            return {"ok": False, "error": "invalid", "pos": self.pos}
        if edge(self.pos, nxt) in self.closed:
            return {"ok": False, "error": "blocked", "pos": self.pos}
        self.pos = nxt
        self.moves += 1
        return {"ok": True, "pos": self.pos}

w = World(dynamic=True)
print("Move A -> B:", w.move("B"))
print("Move B -> D:", w.move("D"))

Move A -> B: {'ok': True, 'pos': 'B'}
Move B -> D: {'ok': False, 'error': 'blocked', 'pos': 'B'}


## 3. Vòng điều khiển agent
Định nghĩa bộ nhớ, mục tiêu và giới hạn số lần gọi công cụ. Một kế hoạch chỉ bị bỏ
khi phản hồi thực tế cho thấy nó không còn thực thi được.

In [4]:
def run_agent(world, goal="F", budget=12):
    blocked, route, log = set(), [], []
    visited = [world.pos]
    calls, plans, status = 0, 0, "budget_exhausted"
    while world.pos != goal and calls < budget:
        if not route:
            path = plan_route(GRAPH, world.pos, goal, blocked)
            plans += 1
            log.append("PLAN " + str(path))
            if path is None:
                status = "unreachable"
                break
            route = path[1:]
        src, nxt = world.pos, route.pop(0)
        observation = world.move(nxt)
        calls += 1
        log.append(f"ACT {src}->{nxt}: {observation}")
        if observation["ok"]:
            visited.append(world.pos)
        else:
            blocked.add(edge(src, nxt))
            route = []
    if world.pos == goal:
        status = "success"
    return dict(status=status, path=visited, calls=calls,
                plans=plans, blocked=blocked, log=log)

print("Agent ready: planner + tool + memory + stopping rule.")

Agent ready: planner + tool + memory + stopping rule.


## 4. Chạy thực nghiệm
So sánh môi trường tĩnh và môi trường có biến cố. Một lần gọi công cụ thất bại
vẫn tiêu tốn ngân sách, nhưng không tăng số bước di chuyển thành công.

In [5]:
baseline = run_agent(World())
dynamic = run_agent(World(dynamic=True))
for line in dynamic["log"]:
    print(line)
for name, result in [("STATIC", baseline), ("DYNAMIC", dynamic)]:
    print(name, result["status"], "path=", "->".join(result["path"]),
          "moves=", len(result["path"]) - 1,
          "calls=", result["calls"], "plans=", result["plans"])
assert dynamic["status"] == "success"
assert dynamic["path"] == ["A", "B", "C", "E", "F"]
assert edge("B", "D") in dynamic["blocked"]

PLAN ['A', 'B', 'D', 'F']
ACT A->B: {'ok': True, 'pos': 'B'}
ACT B->D: {'ok': False, 'error': 'blocked', 'pos': 'B'}
PLAN ['B', 'C', 'E', 'F']
ACT B->C: {'ok': True, 'pos': 'C'}
ACT C->E: {'ok': True, 'pos': 'E'}
ACT E->F: {'ok': True, 'pos': 'F'}
STATIC success path= A->B->D->F moves= 3 calls= 3 plans= 1
DYNAMIC success path= A->B->C->E->F moves= 4 calls= 5 plans= 2


## 5. Kiểm tra các tình huống có ý nghĩa
Các assert kiểm tra thành công, tái lập kế hoạch, hết ngân sách, không thể đến đích
và trạng thái đã ở đích. Đây là 5 ca xác định, không phải ước lượng xác suất thành công.

In [6]:
cases = {
    "static": (baseline, "success", 3),
    "dynamic": (dynamic, "success", 5),
    "budget": (run_agent(World(dynamic=True), budget=1),
               "budget_exhausted", 1),
    "unreachable": (run_agent(World(closed=[edge("A", "B"),
                       edge("A", "C")])), "unreachable", 2),
    "already_goal": (run_agent(World(start="F")), "success", 0),
}
for name, (result, expected, calls) in cases.items():
    assert result["status"] == expected, (name, result)
    assert result["calls"] == calls, (name, result)
    assert result["calls"] <= 12
    print(f"PASS {name:13} status={expected:17} calls={calls}")
assert dynamic["plans"] == 2
print("5/5 cases passed; dynamic run replanned exactly once.")

PASS static        status=success           calls=3
PASS dynamic       status=success           calls=5
PASS budget        status=budget_exhausted  calls=1
PASS unreachable   status=unreachable       calls=2
PASS already_goal  status=success           calls=0
5/5 cases passed; dynamic run replanned exactly once.


## Nhận xét
Tĩnh: A–B–D–F (3 bước, 3 lần gọi). Động: A–B–C–E–F (4 bước, 5 lần gọi,
2 lần lập kế hoạch). Phản hồi B–D bị chặn làm thay đổi bộ nhớ và kế hoạch.
Agent thích nghi bằng cập nhật trạng thái, **không cập nhật trọng số hoặc học chính sách**.
Giới hạn: đồ thị nhỏ, chi phí bằng nhau, lỗi xác định, cạnh chặn không mở lại.
Muốn tối ưu chi phí khác nhau cần Dijkstra/A*; muốn dùng ngôn ngữ tự nhiên có thể thêm LLM
nhưng phải kiểm tra tham số công cụ và giữ tiêu chí dừng độc lập.